In [23]:
# =========================================================
# 01_cleaning.py — المرحلة 1: قراءة وتنظيف البيانات
# =========================================================

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, trim, lower, upper,
    to_timestamp, from_unixtime,
    count, isnan, isnull
)
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, LongType
)
import os

print("✅ Imports OK")

✅ Imports OK


In [24]:
# =========================================================
# إنشاء SparkSession مع إعدادات Kafka
# =========================================================

spark = SparkSession.builder \
    .appName("UNSW-NB15-Cleaning") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print(f"✅ Spark version: {spark.version}")
print(f"✅ App Name: {spark.sparkContext.appName}")

✅ Spark version: 3.1.2
✅ App Name: UNSW-NB15-Cleaning


In [25]:
# =========================================================
# اختبار الاتصال بـ Kafka
# =========================================================

kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka1:19092,kafka2:19093,kafka3:19094") \
    .option("subscribe", "unsw-nb15-v2") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()

print("✅ Kafka connection established")
print(f"Schema:")
kafka_df.printSchema()

✅ Kafka connection established
Schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



In [26]:
# =========================================================
# قراءة البيانات كـ Batch (بدل Streaming)
# =========================================================

# نقرأ كل الرسائل الموجودة حالياً في Kafka
df_raw = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka1:19092,kafka2:19093,kafka3:19094") \
    .option("subscribe", "unsw-nb15-v2") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print(f"✅ عدد السجلات في Kafka: {df_raw.count()}")
df_raw.printSchema()
df_raw.selectExpr("CAST(value AS STRING)").show(3, truncate=False)

✅ عدد السجلات في Kafka: 2544
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [27]:
# =========================================================
# Schema الكامل (49 عمود)
# =========================================================

schema = StructType([
    StructField("srcip", StringType(), True),
    StructField("sport", StringType(), True),
    StructField("dstip", StringType(), True),
    StructField("dsport", StringType(), True),
    StructField("proto", StringType(), True),
    StructField("state", StringType(), True),
    StructField("dur", StringType(), True),
    StructField("sbytes", StringType(), True),
    StructField("dbytes", StringType(), True),
    StructField("sttl", StringType(), True),
    StructField("dttl", StringType(), True),
    StructField("sloss", StringType(), True),
    StructField("dloss", StringType(), True),
    StructField("service", StringType(), True),
    StructField("Sload", StringType(), True),
    StructField("Dload", StringType(), True),
    StructField("Spkts", StringType(), True),
    StructField("Dpkts", StringType(), True),
    StructField("swin", StringType(), True),
    StructField("dwin", StringType(), True),
    StructField("stcpb", StringType(), True),
    StructField("dtcpb", StringType(), True),
    StructField("smeansz", StringType(), True),
    StructField("dmeansz", StringType(), True),
    StructField("trans_depth", StringType(), True),
    StructField("res_bdy_len", StringType(), True),
    StructField("Sjit", StringType(), True),
    StructField("Djit", StringType(), True),
    StructField("Stime", StringType(), True),
    StructField("Ltime", StringType(), True),
    StructField("Sintpkt", StringType(), True),
    StructField("Dintpkt", StringType(), True),
    StructField("tcprtt", StringType(), True),
    StructField("synack", StringType(), True),
    StructField("ackdat", StringType(), True),
    StructField("is_sm_ips_ports", StringType(), True),
    StructField("ct_state_ttl", StringType(), True),
    StructField("ct_flw_http_mthd", StringType(), True),
    StructField("is_ftp_login", StringType(), True),
    StructField("ct_ftp_cmd", StringType(), True),
    StructField("ct_srv_src", StringType(), True),
    StructField("ct_srv_dst", StringType(), True),
    StructField("ct_dst_ltm", StringType(), True),
    StructField("ct_src_ltm", StringType(), True),
    StructField("ct_src_dport_ltm", StringType(), True),
    StructField("ct_dst_sport_ltm", StringType(), True),
    StructField("ct_dst_src_ltm", StringType(), True),
    StructField("attack_cat", StringType(), True),
    StructField("Label", StringType(), True),
])

print(f"✅ Schema defined: {len(schema.fields)} columns")

✅ Schema defined: 49 columns


In [28]:
# =========================================================
# تحويل البيانات من Kafka إلى String
# =========================================================

from pyspark.sql.functions import from_csv, col

# نحوّل قيمة Kafka (Binary) إلى String
df_string = df_raw.selectExpr("CAST(value AS STRING) as csv_line")

print(f"✅ عدد السطور: {df_string.count()}")
df_string.show(5, truncate=False)

✅ عدد السطور: 2544
+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [29]:
# =========================================================
# تقسيم CSV إلى 49 عمود
# =========================================================

# نقسم السطر على الفاصلة
from pyspark.sql.functions import split

df_split = df_string.withColumn("cols", split(col("csv_line"), ","))

print(f"✅ تم التقسيم")
df_split.select("csv_line").show(2, truncate=True)
print(f"عدد الأعمدة بعد التقسيم: {len(df_split.select('cols').first()['cols'])}")

✅ تم التقسيم
+--------------------+
|            csv_line|
+--------------------+
|59.166.0.0,6055,1...|
|59.166.0.3,19852,...|
+--------------------+
only showing top 2 rows

عدد الأعمدة بعد التقسيم: 48001


In [30]:
# =========================================================
# تطبيق Schema على الأعمدة
# =========================================================

# نحوّل المصفوفة إلى 49 عمود
df_columns = df_split.select(
    *[col("cols").getItem(i).alias(schema.fields[i].name) for i in range(49)]
)

print(f"✅ تم تطبيق Schema")
df_columns.printSchema()
df_columns.show(3, truncate=False)

✅ تم تطبيق Schema
root
 |-- srcip: string (nullable = true)
 |-- sport: string (nullable = true)
 |-- dstip: string (nullable = true)
 |-- dsport: string (nullable = true)
 |-- proto: string (nullable = true)
 |-- state: string (nullable = true)
 |-- dur: string (nullable = true)
 |-- sbytes: string (nullable = true)
 |-- dbytes: string (nullable = true)
 |-- sttl: string (nullable = true)
 |-- dttl: string (nullable = true)
 |-- sloss: string (nullable = true)
 |-- dloss: string (nullable = true)
 |-- service: string (nullable = true)
 |-- Sload: string (nullable = true)
 |-- Dload: string (nullable = true)
 |-- Spkts: string (nullable = true)
 |-- Dpkts: string (nullable = true)
 |-- swin: string (nullable = true)
 |-- dwin: string (nullable = true)
 |-- stcpb: string (nullable = true)
 |-- dtcpb: string (nullable = true)
 |-- smeansz: string (nullable = true)
 |-- dmeansz: string (nullable = true)
 |-- trans_depth: string (nullable = true)
 |-- res_bdy_len: string (nullable = true)


In [31]:
# =========================================================
# تنظيف القيم الناقصة
# =========================================================

from pyspark.sql.functions import when, trim

df_clean = df_columns

# استبدل "-" و "" بـ null
for field in schema.fields:
    col_name = field.name
    df_clean = df_clean.withColumn(
        col_name,
        when(
            (col(col_name) == "-") | (trim(col(col_name)) == ""),
            None
        ).otherwise(trim(col(col_name)))
    )

print(f"✅ تم تنظيف القيم الناقصة")
df_clean.show(3, truncate=False)

✅ تم تنظيف القيم الناقصة
+----------+-----+-------------+------+-----+-----+--------+------+------+----+----+-----+-----+-------+-----------+-----------+-----+-----+----+----+----------+----------+-------+-------+-----------+-----------+---------+---------+----------+----------+--------+--------+--------+--------+--------+---------------+------------+----------------+------------+----------+----------+----------+----------+----------+----------------+----------------+--------------+----------+-------------+
|srcip     |sport|dstip        |dsport|proto|state|dur     |sbytes|dbytes|sttl|dttl|sloss|dloss|service|Sload      |Dload      |Spkts|Dpkts|swin|dwin|stcpb     |dtcpb     |smeansz|dmeansz|trans_depth|res_bdy_len|Sjit     |Djit     |Stime     |Ltime     |Sintpkt |Dintpkt |tcprtt  |synack  |ackdat  |is_sm_ips_ports|ct_state_ttl|ct_flw_http_mthd|is_ftp_login|ct_ftp_cmd|ct_srv_src|ct_srv_dst|ct_dst_ltm|ct_src_ltm|ct_src_dport_ltm|ct_dst_sport_ltm|ct_dst_src_ltm|attack_cat|Label        |

In [32]:
# =========================================================
# تحويل الأنواع
# =========================================================

from pyspark.sql.types import IntegerType, DoubleType

# قائمة الأعمدة الرقمية (Integer)
int_cols = [
    "sport", "dsport", "sbytes", "dbytes", "sttl", "dttl",
    "sloss", "dloss", "Spkts", "Dpkts", "swin", "dwin",
    "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth",
    "res_bdy_len", "Stime", "Ltime", "is_sm_ips_ports",
    "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login",
    "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm",
    "ct_src_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "Label"
]

# قائمة الأعمدة العشرية (Double)
double_cols = [
    "dur", "Sload", "Dload", "Sjit", "Djit",
    "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat"
]

# تحويل
for c in int_cols:
    df_clean = df_clean.withColumn(c, col(c).cast(IntegerType()))

for c in double_cols:
    df_clean = df_clean.withColumn(c, col(c).cast(DoubleType()))

print(f"✅ تم تحويل الأنواع")
df_clean.printSchema()

✅ تم تحويل الأنواع
root
 |-- srcip: string (nullable = true)
 |-- sport: integer (nullable = true)
 |-- dstip: string (nullable = true)
 |-- dsport: integer (nullable = true)
 |-- proto: string (nullable = true)
 |-- state: string (nullable = true)
 |-- dur: double (nullable = true)
 |-- sbytes: integer (nullable = true)
 |-- dbytes: integer (nullable = true)
 |-- sttl: integer (nullable = true)
 |-- dttl: integer (nullable = true)
 |-- sloss: integer (nullable = true)
 |-- dloss: integer (nullable = true)
 |-- service: string (nullable = true)
 |-- Sload: double (nullable = true)
 |-- Dload: double (nullable = true)
 |-- Spkts: integer (nullable = true)
 |-- Dpkts: integer (nullable = true)
 |-- swin: integer (nullable = true)
 |-- dwin: integer (nullable = true)
 |-- stcpb: integer (nullable = true)
 |-- dtcpb: integer (nullable = true)
 |-- smeansz: integer (nullable = true)
 |-- dmeansz: integer (nullable = true)
 |-- trans_depth: integer (nullable = true)
 |-- res_bdy_len: integer

In [22]:
pip install pandas numpy pyarrow

Defaulting to user installation because normal site-packages is not writeable
     |################################| 9.5 MB 167 kB/s            
     |################################| 14.8 MB 108 kB/s            
     |################################| 25.6 MB 213 kB/s            
Note: you may need to restart the kernel to use updated packages.


In [33]:
# حفظ CSV عبر Pandas
import pandas as pd

df_pandas = df_clean.toPandas()

df_pandas.to_csv(
    "/workspace/data/curated/unsw_nb15_cleaned.csv",
    index=False,
    encoding="utf-8"
)

print(f"✅ عدد السجلات: {len(df_pandas)}")
print("✅ تم الحفظ: unsw_nb15_cleaned.csv")

✅ عدد السجلات: 2544
✅ تم الحفظ: unsw_nb15_cleaned.csv


In [34]:
# =========================================================
# 1. تحقق من Kafka
# =========================================================

df_check = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka1:19092,kafka2:19093,kafka3:19094") \
    .option("subscribe", "unsw-nb15-v2") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load()

total = df_check.count()
print(f"📊 عدد السجلات في Kafka: {total}")

📊 عدد السجلات في Kafka: 2544


In [35]:
# =========================================================
# تحقق: كم سطر فعلي في Kafka؟
# =========================================================

# نقرأ أول 3 رسائل فقط لفحصها
df_sample = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka1:19092,kafka2:19093,kafka3:19094") \
    .option("subscribe", "unsw-nb15-v2") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load() \
    .selectExpr("CAST(value AS STRING) as csv_batch")

# نأخذ أول رسالة
sample = df_sample.first()
if sample:
    csv_content = sample['csv_batch']
    lines = csv_content.split('\n')
    print(f"✅ عدد السطور في أول رسالة: {len(lines)}")
    print(f"✅ أول سطر: {lines[0][:100]}...")
    print(f"✅ آخر سطر: {lines[-1][:100]}...")

✅ عدد السطور في أول رسالة: 1001
✅ أول سطر: 59.166.0.0,6055,149.171.126.5,54145,tcp,FIN,0.072974,4238,60788,31,29,7,30,-,458245.4375,6571546.5,7...
✅ آخر سطر: ...


In [36]:
# =========================================================
# قراءة كل Kafka + تفجير الرسائل إلى سطور
# =========================================================

from pyspark.sql.functions import explode, split, col

# 1. قراءة Kafka
df_batches = spark.read \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka1:19092,kafka2:19093,kafka3:19094") \
    .option("subscribe", "unsw-nb15-v2") \
    .option("startingOffsets", "earliest") \
    .option("endingOffsets", "latest") \
    .load() \
    .selectExpr("CAST(value AS STRING) as batch")

# 2. تفجير كل رسالة إلى سطور
df_lines = df_batches \
    .withColumn("line", explode(split(col("batch"), "\n"))) \
    .select("line") \
    .filter(col("line") != "")

print(f"📊 عدد السطور الإجمالي: {df_lines.count()}")

📊 عدد السطور الإجمالي: 2540047


In [37]:
# =========================================================
# تطبيق 49 عمود Schema
# =========================================================

from pyspark.sql.functions import split, col

df_split = df_lines.withColumn("cols", split(col("line"), ","))

df_columns = df_split.select(
    *[col("cols").getItem(i).alias(schema.fields[i].name) for i in range(49)]
)

print(f"✅ تم تطبيق Schema")
df_columns.cache()
print(f"📊 عدد السجلات: {df_columns.count()}")

✅ تم تطبيق Schema
📊 عدد السجلات: 2540047


In [38]:
# =========================================================
# تنظيف القيم "-" و ""
# =========================================================

from pyspark.sql.functions import when, trim

df_clean = df_columns

for field in schema.fields:
    c = field.name
    df_clean = df_clean.withColumn(
        c,
        when(
            (col(c) == "-") | (trim(col(c)) == ""),
            None
        ).otherwise(trim(col(c)))
    )

print(f"✅ تم تنظيف القيم")

✅ تم تنظيف القيم


In [39]:
# =========================================================
# تحويل الأنواع
# =========================================================

from pyspark.sql.types import IntegerType, DoubleType

int_cols = [
    "sport", "dsport", "sbytes", "dbytes", "sttl", "dttl",
    "sloss", "dloss", "Spkts", "Dpkts", "swin", "dwin",
    "stcpb", "dtcpb", "smeansz", "dmeansz", "trans_depth",
    "res_bdy_len", "Stime", "Ltime", "is_sm_ips_ports",
    "ct_state_ttl", "ct_flw_http_mthd", "is_ftp_login",
    "ct_ftp_cmd", "ct_srv_src", "ct_srv_dst", "ct_dst_ltm",
    "ct_src_ltm", "ct_src_dport_ltm", "ct_dst_sport_ltm",
    "ct_dst_src_ltm", "Label"
]

double_cols = [
    "dur", "Sload", "Dload", "Sjit", "Djit",
    "Sintpkt", "Dintpkt", "tcprtt", "synack", "ackdat"
]

for c in int_cols:
    df_clean = df_clean.withColumn(c, col(c).cast(IntegerType()))

for c in double_cols:
    df_clean = df_clean.withColumn(c, col(c).cast(DoubleType()))

print(f"✅ تم تحويل الأنواع")
df_clean.cache()
print(f"📊 عدد السجلات: {df_clean.count()}")

✅ تم تحويل الأنواع
📊 عدد السجلات: 2540047


In [42]:
# احفظ Parquet في /tmp (داخل الحاوية)
df_clean.coalesce(1).write.mode("overwrite").parquet("file:///tmp/unsw_cleaned.parquet")
print("✅ تم الحفظ في /tmp")

✅ تم الحفظ في /tmp
